In [ ]:
# === ノートブック共通の前処理 (llm_math パッケージの読み込み) ===
import sys
from pathlib import Path

# llm_math パッケージの候補パス
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 親ディレクトリも候補に追加 (notebooks/ フォルダで実行する場合)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math の import を試行
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[注意] llm_math パッケージの読み込み テキスト: {e}")
    print("  GitHub リポジトリを clone して colab_setup.sh を実行してください。")
# === 前処理ここまで ===


# Ch 13. テキスト — テキスト ベクトル 空間テキスト

> **学習目標**
> - Word2Vec(Skip-gram, CBOW)テキスト テキスト 関数テキスト テキスト度テキスト
> - GloVeテキスト テキスト 行列 テキスト テキスト
> - テキスト 学習 テキスト テキスト テキスト テキスト

## 13.1 テキスト テキスト テキスト

"テキスト テキスト テキスト テキスト テキスト テキスト テキスト" (Firth, 1957).

**テキスト テキスト(distributed representation)**: テキスト テキスト次元 ベクトルテキスト テキスト 次元テキスト テキスト エンコード.
- "king" = [0.9, 0.7, 0.1, ...] (テキスト, テキスト, ...)

## 13.2 Word2Vec — Skip-gram

テキスト テキスト テキスト テキスト テキスト:
$$\max_\theta \sum_{t=1}^{T} \sum_{-c \leq j \leq c, j \neq 0} \log P(w_{t+j} | w_t; \theta)$$

テキスト モデル:
$$P(w_O | w_I) = \frac{\exp(\mathbf{v}'_{w_O}{}^\top \mathbf{v}_{w_I})}{\sum_{w \in V} \exp(\mathbf{v}'_w{}^\top \mathbf{v}_{w_I})}$$

- $\mathbf{v}_w$: テキスト $w$テキスト テキスト ベクトル
- $\mathbf{v}'_w$: テキスト $w$テキスト 出力 ベクトル
- テキスト 計算テキスト $O(|V|)$テキスト テキスト → **Negative Sampling** テキスト


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# Skip-gram with Negative Sampling (テキスト)
class SkipGramNS:
    def __init__(self, vocab_size, embed_dim=50, seed=0):
        rng = np.random.default_rng(seed)
        self.W_in = rng.standard_normal((vocab_size, embed_dim)) * 0.01
        self.W_out = rng.standard_normal((vocab_size, embed_dim)) * 0.01
        self.embed_dim = embed_dim

    def forward(self, center_idx, context_idx, neg_indices):
        v = self.W_in[center_idx]      # (D,)
        u_pos = self.W_out[context_idx]  # (D,)
        u_neg = self.W_out[neg_indices]  # (K, D)
        return v, u_pos, u_neg

    def loss_and_grad(self, center_idx, context_idx, neg_indices):
        v = self.W_in[center_idx]
        u_pos = self.W_out[context_idx]
        u_neg = self.W_out[neg_indices]  # (K, D)

        # positive: log sigma(u_pos . v)
        score_pos = u_pos @ v
        sigma_pos = 1 / (1 + np.exp(-score_pos))
        loss = -np.log(sigma_pos + 1e-12)

        # negative: log sigma(-u_neg . v)
        score_neg = u_neg @ v  # (K,)
        sigma_neg = 1 / (1 + np.exp(-score_neg))
        loss -= np.sum(np.log(1 - sigma_neg + 1e-12))

        # テキスト
        dloss_dscore_pos = sigma_pos - 1
        dloss_dscore_neg = sigma_neg  # (K,)

        grad_v = dloss_dscore_pos * u_pos + np.sum(dloss_dscore_neg[:, None] * u_neg, axis=0)
        grad_u_pos = dloss_dscore_pos * v
        grad_u_neg = dloss_dscore_neg[:, None] * v[None, :]

        return loss, grad_v, grad_u_pos, grad_u_neg

    def train_step(self, center_idx, context_idx, neg_indices, lr=0.025):
        loss, gv, gu_pos, gu_neg = self.loss_and_grad(center_idx, context_idx, neg_indices)
        self.W_in[center_idx] -= lr * gv
        self.W_out[context_idx] -= lr * gu_pos
        self.W_out[neg_indices] -= lr * gu_neg
        return loss

# テキスト テキスト
corpus_text = "the king killed the queen and the prince was sad and the kingdom fell"
words = corpus_text.split()
vocab = sorted(set(words))
word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for w, i in word_to_idx.items()}
vocab_size = len(vocab)
print(f"Vocabulary Size: {vocab_size}, テキスト Length: {len(words)}")

# 学習 データ テキスト (skip-gram pairs)
window = 2
pairs = []
for i, w in enumerate(words):
    for j in range(max(0, i - window), min(len(words), i + window + 1)):
        if j != i:
            pairs.append((word_to_idx[w], word_to_idx[words[j]]))
print(f"Training テキスト テキスト: {len(pairs)}")

# テキスト テキスト 分布 (unigram^0.75)
word_freqs = Counter(words)
neg_dist = np.array([word_freqs[w]**0.75 for w in vocab])
neg_dist /= neg_dist.sum()

# 学習
model = SkipGramNS(vocab_size, embed_dim=20, seed=0)
losses = []
for epoch in range(50):
    total_loss = 0
    np.random.shuffle(pairs)
    for center, context in pairs:
        K = 5
        negs = np.random.choice(vocab_size, size=K, p=neg_dist)
        total_loss += model.train_step(center, context, negs, lr=0.05)
    losses.append(total_loss / len(pairs))

plt.figure(figsize=(9, 4))
plt.plot(losses, 'b-')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Skip-gram with Negative Sampling Learning Curve')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch13_word2vec_loss.png', dpi=100, bbox_inches='tight')
plt.show()


## 13.3 テキスト テキスト (Word Analogy)

Word2Vecテキスト テキスト 結果:
$$\mathbf{v}(\text{king}) - \mathbf{v}(\text{man}) + \mathbf{v}(\text{woman}) \approx \mathbf{v}(\text{queen})$$

ベクトル 空間テキスト テキスト, テキスト テキスト テキスト 次元テキスト 学習テキスト.


In [ ]:
# テキスト テキスト度 計算 (テキスト モデル)
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12)

# 学習テキスト テキスト テキスト度
print("Trainingテキスト word Embedding Similarity:")
for w1 in ['king', 'queen', 'kingdom']:
    sims = []
    for w2 in vocab:
        if w1 != w2:
            v1 = model.W_in[word_to_idx[w1]]
            v2 = model.W_in[word_to_idx[w2]]
            sims.append((w2, cosine_sim(v1, v2)))
    sims.sort(key=lambda x: -x[1])
    print(f"  '{w1}'テキスト テキスト テキスト word: {sims[:3]}")


## 13.4 GloVe — テキスト テキスト テキスト

GloVe (Pennington et al., 2014)テキスト テキスト テキスト テキスト(co-occurrence) テキスト テキスト 学習.

$$J = \sum_{i,j} f(X_{ij}) \left(\mathbf{w}_i^\top \tilde{\mathbf{w}}_j + b_i + \tilde{b}_j - \log X_{ij}\right)^2$$

- $X_{ij}$: テキスト $i$テキスト $j$テキスト テキスト テキスト
- $f(X_{ij})$: テキスト 関数 (テキスト テキスト テキスト テキスト テキスト テキスト)
- $\mathbf{w}, \tilde{\mathbf{w}}$: テキスト テキスト テキスト ベクトル (テキスト テキスト)

Word2Vecテキスト テキスト テキスト テキスト テキスト. テキスト テキスト テキスト テキスト.


In [ ]:
# テキスト 行列 計算 (テキスト)
def build_cooc_matrix(words, window=2, vocab_size=None, word_to_idx=None):
    """word テキスト Matrix."""
    if vocab_size is None:
        vocab = sorted(set(words))
        word_to_idx = {w: i for i, w in enumerate(vocab)}
        vocab_size = len(vocab)
    M = np.zeros((vocab_size, vocab_size))
    for i, w in enumerate(words):
        wi = word_to_idx[w]
        for j in range(max(0, i - window), min(len(words), i + window + 1)):
            if j != i:
                wj = word_to_idx[words[j]]
                # Distanceテキスト テキスト Weight
                weight = 1.0 / abs(i - j)
                M[wi, wj] += weight
    return M, word_to_idx

M, w2i = build_cooc_matrix(words, window=2, vocab_size=vocab_size, word_to_idx=word_to_idx)
print(f"テキスト 行列: {M.shape}")
print(f"テキスト 5x5:")
print(M[:5, :5].round(2))
print(f"\n'king'テキスト テキスト テキスト テキスト word:")
king_idx = word_to_idx['king']
cooc = [(idx_to_word[i], M[king_idx, i]) for i in range(vocab_size) if i != king_idx]
cooc.sort(key=lambda x: -x[1])
for w, c in cooc[:5]:
    print(f"  {w}: {c:.2f}")


## 13.5 テキスト 学習 テキスト テキスト テキスト

Word2Vec/GloVeテキスト **テキスト テキスト** — テキスト テキスト テキスト テキスト ベクトル.

問題: "bank" (テキスト)テキスト "bank" (テキスト)テキスト テキスト ベクトル → **テキスト 問題**.

テキスト **学習 テキスト テキスト 行列** $\mathbf{E} \in \mathbb{R}^{|V| \times d}$:
- テキスト ID $i$ → $\mathbf{E}[i]$
- テキスト テキスト Attentionテキスト テキスト テキスト テキスト テキスト
- "bank"テキスト テキスト テキスト テキスト, Attention テキスト テキスト テキスト テキスト

テキスト **テキスト テキスト**テキスト テキスト度テキスト テキスト (Positional Encoding, Ch 16テキスト テキスト).

テキスト テキスト: テキスト テキスト テキスト $\sqrt{d}$テキスト テキスト.
$$\mathbf{E}' = \mathbf{E} \cdot \sqrt{d}$$


In [ ]:
# テキスト テキスト (PyTorch)
import torch
import torch.nn as nn

vocab_size = 1000
embed_dim = 64

# 学習 テキスト テキスト
embedding = nn.Embedding(vocab_size, embed_dim)
print(f"Embedding Matrix: {embedding.weight.shape}")
print(f"Parameter Count: {embedding.weight.numel():,}")

# テキスト
seq = torch.tensor([[1, 5, 10, 25, 100]])  # (batch, seq_len)
emb = embedding(seq)  # (batch, seq_len, embed_dim)
emb_scaled = emb * (embed_dim ** 0.5)

print(f"\nEmbedding Output: {emb.shape}")
print(f"Scaling テキスト std: {emb.std():.4f}")
print(f"Scaling テキスト std: {emb_scaled.std():.4f} (≈ {embed_dim ** 0.5:.2f}テキスト)")

# 可視化: テキスト ベクトル (テキスト 20次元)
fig, ax = plt.subplots(figsize=(12, 4))
emb_sample = embedding.weight[:20, :20].detach().numpy()
im = ax.imshow(emb_sample, cmap='RdBu', aspect='auto')
plt.colorbar(im, ax=ax)
ax.set_xlabel('Embedding Dimension')
ax.set_ylabel('テキスト ID')
ax.set_title('Embedding Matrix (テキスト 20x20)')
plt.tight_layout()
plt.savefig('../figures/ch13_embedding_matrix.png', dpi=100, bbox_inches='tight')
plt.show()


## 13.6 テキスト 可視化 — t-SNE, PCA

テキスト次元 テキスト 2Dテキスト テキスト 可視化. テキスト テキスト テキスト テキスト.


In [ ]:
# テキスト 可視化 (テキスト PCA)
from sklearn.decomposition import PCA

# 学習テキスト テキスト テキスト
embeddings = model.W_in  # (vocab_size, embed_dim)
print(f"Embedding: {embeddings.shape}")

# PCAテキスト 2D テキスト
pca = PCA(n_components=2)
emb_2d = pca.fit_transform(embeddings)

# 可視化
fig, ax = plt.subplots(figsize=(8, 7))
for i, word in enumerate(vocab):
    ax.scatter(emb_2d[i, 0], emb_2d[i, 1], c='blue', s=80)
    ax.annotate(word, (emb_2d[i, 0], emb_2d[i, 1]), fontsize=11)
ax.set_title('Word2Vec Embedding (PCA 2D テキスト)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch13_embedding_tsne.png', dpi=100, bbox_inches='tight')
plt.show()
print("テキスト wordテキスト テキスト テキスト テキスト (テキスト テキスト Patternテキスト テキスト テキスト テキスト).")


## 13.7 要点

| モデル | テキスト | テキスト |
|---|---|---|
| Word2Vec | テキスト テキスト (skip-gram/CBOW) | テキスト テキスト |
| GloVe | テキスト テキスト | テキスト テキスト テキスト |
| FastText | テキスト Word2Vec | OOV テキスト テキスト |
| Transformer テキスト | 学習 テキスト 行列 | テキスト テキスト テキスト |

## 演習問題

1. Skip-gramテキスト CBOWテキスト テキスト テキスト, テキスト テキスト テキスト テキスト テキスト.
2. テキスト テキスト テキスト $K = 1, 5, 15$テキスト テキスト 学習 結果テキスト 比較テキスト.
3. GloVeテキスト テキスト 関数 $f(x) = (x/x_{\max})^\alpha$ if $x < x_{\max}$ else 1 テキスト テキスト テキスト.
4. Word2Vecテキスト テキスト テキスト テキスト テキスト テキスト 結果テキスト テキスト.
5. テキスト テキスト $\sqrt{d}$ テキスト テキスト テキスト Attention テキスト テキスト テキスト.

> 解答: `solutions/ch13_solutions.ipynb`
